# L11 — Distributed serving: tensor parallel & autoscaling on Ray
**Objective 14.**

**Northfield Grocers context:** Saturday 10:00 traffic on the customer assistant is 4× Tuesday 15:00. Northfield needs the dense 31B model served TP=4, an autoscaler that pre-warms before known peaks and keeps two replicas for availability, and proof the service survives losing a worker mid-peak.

**Retail use cases:** Peak-hour scaling for the customer assistant; high availability for the store-ops assistant during trading hours.

**Platform:** Databricks multi-GPU (NC96ads_A100_v4, 4× A100). Steps 1–3 are NumPy; Step 4 needs the cluster.

**Done means:** sharded matmul equals full within 1e-6; autoscaler simulation keeps p95 queue time under the SLA; TP=4 endpoint survives a simulated worker loss.

## Step 1 — Tensor parallelism by hand
*Why:* TP splits a weight matrix across GPUs. Column-parallel: each GPU computes part of the output columns, then concatenate. Row-parallel: each GPU holds part of the input dimension, then all-reduce (sum). Do both and prove they equal the unsharded matmul.

In [ ]:
# === Dependency check (no installs, no restarts — see L00_setup for one-time cluster setup) ===
import importlib.util
_REQ = ["numpy", "pandas", "sklearn", "onnx", "onnxruntime", "skl2onnx"]
_missing = [m for m in _REQ if importlib.util.find_spec(m) is None]
if _missing:
    raise ImportError(f"Missing packages {_missing}. Run labs/L00_Setup/L00_setup.ipynb once on this cluster/venv "
                      f"(or `%pip install {' '.join(_missing)}` in a new cell, then restart Python and Run All from the top).")
print("All lab dependencies present.")

In [ ]:
# === Lab environment header (identical in every lab) ===
import os, sys, json, time, math, shutil, re
import numpy as np, pandas as pd
# Mode: "GPU" runs the full lab on Azure GPU compute; "SMOKE" runs the CPU/synthetic path anywhere.
LAB_MODE = os.environ.get("LAB_MODE") or ("GPU" if shutil.which("nvidia-smi") else "SMOKE")
# Data folder: env override → package-relative (../../data) → Databricks Unity Catalog volume
_candidates = [os.environ.get("DATA_DIR"), os.path.abspath(os.path.join(os.getcwd(), "..", "..", "data")), "/Volumes/northfield/llmops/labdata"]
DATA_DIR = next((c for c in _candidates if c and os.path.exists(os.path.join(c, "catalog_items.csv"))), None)
if DATA_DIR is None:
    raise FileNotFoundError("Lab data not found. Set os.environ['DATA_DIR'] to the folder containing catalog_items.csv "
                            "(e.g. a Unity Catalog volume path) in a cell above this one.")
def gpu_only(msg):
    """Called wherever a step needs a GPU / model download that the smoke path cannot run."""
    print(f"[{LAB_MODE}] GPU-only step not executed here: {msg}")
def check(cond, msg):
    """Binary 'done means' assertion — prints PASS/FAIL and raises on FAIL so the notebook stops."""
    print(("PASS " if cond else "FAIL ") + msg); assert cond, msg
print(f"LAB_MODE={LAB_MODE}  DATA_DIR={DATA_DIR}")

In [ ]:
rng = np.random.default_rng(0); x = rng.normal(size=(8, 1024)).astype(np.float64); W = rng.normal(size=(1024, 4096)); TP = 4
full = x @ W

def column_parallel(x, W, tp):
    # TODO: split W along axis=1; each shard computes x@W_i; concatenate
    raise NotImplementedError('complete this step')

def row_parallel(x, W, tp):
    # TODO: split x along axis=1 and W along axis=0; partial = x_i@W_i; all-reduce = sum
    raise NotImplementedError('complete this step')

check(np.allclose(column_parallel(x, W, TP), full, atol=1e-6) and np.allclose(row_parallel(x, W, TP), full, atol=1e-6), "both shardings equal the full matmul")
print("bytes per GPU for W:", W.nbytes // TP, "vs full", W.nbytes)

## Step 2 — What TP costs: the all-reduce
*Why:* row-parallel needs a sum across GPUs every layer — that is why NVLink SKUs matter (L01). Count the communication volume per layer for the 8×4096 activation and compare it to the compute saved.

In [ ]:
act_bytes = 8 * 4096 * 2   # bf16 activation
allreduce_bytes = 2 * (TP - 1) / TP * act_bytes   # ring all-reduce volume per GPU
print(f"all-reduce per layer per GPU ≈ {allreduce_bytes/1e3:.1f} KB; ×60 layers per token step")
check(allreduce_bytes > 0, "communication cost quantified")

## Step 3 — Autoscaling simulation
*Why:* before touching a real autoscaler, simulate one. Arrivals follow a daily curve; each replica serves `cap` requests/s; scale up when queue p95 exceeds the SLA, scale down when utilisation is low, with a minimum of 2 replicas for HA. At minute 300 a replica dies.

In [ ]:
def simulate(minutes=720, cap=20, sla_s=2.0, min_rep=2, max_rep=8, fail_at=300, scale_up_cooldown=5):
    # TODO: per-minute loop: arrivals = 60*rate(t); served = 60*cap*replicas; queue update; wait = queue/(cap*replicas); scaling with cooldown; drop replica at fail_at
    raise NotImplementedError('complete this step')

sim = simulate(); p95_wait = sim.wait_s.quantile(.95)
print(f"replicas min/max {sim.replicas.min()}/{sim.replicas.max()}; p95 queue wait {p95_wait:.2f}s; wait at failure+1 min {sim.wait_s[301]:.2f}s")
check(p95_wait < 2.0 and sim.replicas.min() >= 1, "autoscaler keeps p95 wait under SLA and recovers from replica loss")

## Step 4 — Real TP=4 serving on Ray-on-Databricks (cluster)
*Why:* `ray.util.spark.setup_ray_cluster` starts Ray across Databricks workers; vLLM with `--tensor-parallel-size 4` shards Gemma 4 31B over the 4 A100s. Then kill one worker process and watch Ray Serve reschedule. Exact flags/APIs are version-sensitive — lab guide §Step 4.

In [ ]:
if LAB_MODE == "GPU":
    gpu_only("Start Ray on Spark, launch vLLM TP=4, run the L03 harness at concurrency 32, simulate worker loss, record recovery time.")
recovery = {"tp4_throughput_tok_s": None, "recovery_s_after_worker_loss": None}
print(recovery); print("L11 complete." + ("" if LAB_MODE == "GPU" else " (SMOKE: Step 4 needs the multi-GPU cluster.)"))